# Sesión 08. Un agente simple con Google ADK

Un **modelo de lenguaje** (LLM) estima una distribución condicional $p_\theta(y \mid x)$: recibe un texto $x$ y genera una continuación $y$ token por token. Por sí solo solo produce texto.

Un **agente** envuelve al modelo en un bucle de control. En cada paso $t$ el agente mira el historial $h_t$ (mensajes, resultados de herramientas, estado) y elige una acción $a_t$ según una política

$$a_t \sim \pi_\theta(a \mid h_t), \qquad a_t \in \{\text{responder}\} \cup \{\text{llamar a } f_1, \dots, f_K\}.$$

Si la acción es una llamada a herramienta, el programa ejecuta la función, obtiene una observación $o_t$ y la agrega al historial: $h_{t+1} = h_t \oplus (a_t, o_t)$. El bucle termina cuando el modelo responde en texto. Este es el patrón **ReAct** (razonar, actuar, observar).

En este cuaderno vamos a:

1. Escribir tres herramientas como funciones de Python con type hints y docstring.
2. Ver el **esquema JSON** que ADK genera para cada una y que recibe el modelo.
3. Construir un `LlmAgent` con un modelo local (`qwen3:4b` en Ollama, vía LiteLLM).
4. Ejecutarlo con `Runner` e `InMemorySessionService`, y leer los **eventos** uno por uno.
5. Usar el **estado de sesión** como memoria entre turnos.
6. Comparar la misma pregunta con y sin herramientas.
7. Abrir el agente en la interfaz de desarrollo `adk web`.

El agente vive en `agentes/asistente_teresa/agent.py` y responde sobre la telenovela *Teresa* (2010) y el calendario del curso. El cuaderno se genera con `scripts/build_agente_notebook.py`; si quieres cambiar una celda, cambia el script y regenera.

## 1. Preparación

Antes de ejecutar, abre el túnel al servidor del laboratorio en otra terminal:

```bash
ssh -N -L 11434:localhost:11434 $GPU_HOST
```

Con el túnel activo, `http://localhost:11434` es el Ollama del servidor. LiteLLM lee esa dirección de la variable `OLLAMA_API_BASE`.

In [1]:
import json
import os
import sys
import time
import urllib.request
from pathlib import Path

os.environ.setdefault("OLLAMA_API_BASE", "http://localhost:11434")


def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / "agentes" / "asistente_teresa").exists():
            return parent
    return p


ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / "agentes"))

with urllib.request.urlopen(os.environ["OLLAMA_API_BASE"] + "/api/tags", timeout=10) as r:
    modelos = sorted(m["name"] for m in json.load(r)["models"])
print("Ollama responde. Modelos con 'qwen':", [m for m in modelos if "qwen" in m])

import google.adk
print("google-adk", google.adk.__version__)

Ollama responde. Modelos con 'qwen': ['qwen2.5-coder:1.5b', 'qwen2.5-coder:7b', 'qwen2.5:0.5b', 'qwen2.5:3b', 'qwen2.5:7b', 'qwen2.5vl:3b', 'qwen3-vl:8b', 'qwen3:4b']
google-adk 2.14.1


## 2. Una herramienta es una función

Para ADK una herramienta es una función de Python común. Tres cosas importan:

- el **nombre** de la función, que el modelo usará para pedirla;
- los **type hints** de los argumentos, que se vuelven tipos del esquema JSON;
- la **docstring**, que se vuelve la descripción. El modelo decide cuándo llamar a la herramienta leyendo esa descripción, así que es parte del *prompt*.

Las herramientas deben ser deterministas y devolver diccionarios con un campo `status`, para que el modelo distinga un resultado válido de un error.

`calcular` evalúa expresiones aritméticas recorriendo el árbol sintáctico con `ast`. Usar `eval` sobre texto que escribió un modelo permitiría ejecutar código arbitrario.

In [2]:
from asistente_teresa import agent as modulo

print(modulo.calcular("7 * 24"))
print(modulo.calcular("(2 + 3) ** 2 / 4"))
print(modulo.calcular("__import__('os').system('ls')"))

{'status': 'ok', 'expresion': '7 * 24', 'resultado': 168}
{'status': 'ok', 'expresion': '(2 + 3) ** 2 / 4', 'resultado': 6.25}
{'status': 'error', 'expresion': "__import__('os').system('ls')", 'motivo': 'elemento no permitido: Call'}


`buscar_personaje` y `consultar_sesion` reciben además un parámetro `tool_context`. ADK lo inyecta al ejecutar la herramienta y lo oculta del esquema: el modelo nunca lo ve. A través de `tool_context.state` la herramienta lee y escribe el **estado de la sesión**.

Para probarlas fuera del agente basta un objeto con un atributo `state`.

In [3]:
class ContextoFalso:
    def __init__(self):
        self.state = {}


ctx = ContextoFalso()
print(modulo.buscar_personaje("aida", ctx))
print(modulo.buscar_personaje("Mariano Sánchez", ctx))
print(modulo.buscar_personaje("Pedro Infante", ctx))
print("estado:", ctx.state)
print(modulo.consultar_sesion(8, ctx))

{'status': 'ok', 'nombre': 'Aída Cervantes', 'actor': 'Ana Brenda Contreras', 'rol': 'Antagonista', 'descripcion': 'Joven de familia acomodada, rival de Teresa en el amor y en lo social.', 'total_personajes_consultados': 1}
{'status': 'ok', 'nombre': 'Mariano Sánchez', 'actor': 'Aarón Díaz', 'rol': 'Coprotagonista', 'descripcion': 'Novio de Teresa desde la juventud, de su mismo barrio, que estudia para salir adelante.', 'total_personajes_consultados': 2}
{'status': 'no_encontrado', 'disponibles': ['Teresa Chávez', 'Arturo de la Barrera', 'Mariano Sánchez', 'Aída Cervantes']}
estado: {'personajes_consultados': ['Aída Cervantes', 'Mariano Sánchez']}
{'status': 'ok', 'numero': 8, 'fecha': '2026-11-21', 'tema': 'Creación de un agente simple con Google ADK', 'hoy': '2026-11-14', 'dias_restantes': 7}


## 3. El esquema JSON que recibe el modelo

ADK envuelve cada función en un `FunctionTool` y arma una `FunctionDeclaration` (nombre, descripción y parámetros en JSON Schema). LiteLLM la traduce al formato de herramientas de OpenAI, que es el que entiende Ollama. Observa que `tool_context` no aparece.

In [4]:
from google.adk.models.lite_llm import _function_declaration_to_tool_param
from google.adk.tools import FunctionTool

decl = FunctionTool(modulo.consultar_sesion)._get_declaration()
print(json.dumps(_function_declaration_to_tool_param(decl), ensure_ascii=False, indent=2))

{
  "type": "function",
  "function": {
    "name": "consultar_sesion",
    "description": "Consulta el calendario del curso de Inteligencia Artificial Aplicada.\n\nÚsala cuando el usuario pregunte por la fecha o el tema de una sesión, o\ncuántos días faltan para ella.\n\nArgs:\n    numero: Número de la sesión, de 1 a 8.\n\nReturns:\n    Un diccionario con status \"ok\", número, fecha (AAAA-MM-DD), tema y\n    dias_restantes (negativo si la sesión ya pasó), o status \"error\".",
    "parameters": {
      "properties": {
        "numero": {
          "title": "Numero",
          "type": "integer"
        }
      },
      "required": [
        "numero"
      ],
      "title": "consultar_sesionParams",
      "type": "object"
    }
  }
}


Con el esquema, el modelo convierte una pregunta en una llamada estructurada. Formalmente, el modelo genera una cadena $c$ que el programa decodifica como $(f, \mathbf{u})$, el nombre de la herramienta y sus argumentos; la llamada es válida si $f$ existe y $\mathbf{u}$ cumple el esquema de $f$. La herramienta devuelve $o = f(\mathbf{u})$.

Todas las herramientas viajan en **cada** petición al modelo y consumen tokens de entrada.

In [5]:
for f in [modulo.buscar_personaje, modulo.calcular, modulo.consultar_sesion]:
    d = FunctionTool(f)._get_declaration()
    texto = json.dumps(_function_declaration_to_tool_param(d), ensure_ascii=False)
    print(f"{d.name:18s} {len(texto):5d} caracteres  parámetros: {list(d.parameters_json_schema['properties'])}")

buscar_personaje     730 caracteres  parámetros: ['nombre']
calcular             645 caracteres  parámetros: ['expresion']
consultar_sesion     629 caracteres  parámetros: ['numero']


## 4. El `LlmAgent`

Un `LlmAgent` reúne cuatro piezas:

| Campo | Papel |
|---|---|
| `model` | El LLM. Con `LiteLlm(model="ollama_chat/qwen3:4b")` usamos un modelo local. |
| `instruction` | El mensaje de sistema. Admite plantillas `{clave?}` que ADK rellena con el estado. |
| `tools` | La lista de funciones disponibles. |
| `name`, `description` | Identifican al agente en los eventos y cuando hay varios agentes. |

Elegimos `qwen3:4b` (2.4 GB) porque en las pruebas encadenó dos herramientas de forma fiable. `qwen2.5:7b` llamó a la primera herramienta y luego hizo la cuenta de memoria.

In [6]:
agente = modulo.construir_agente()
print("nombre:", agente.name)
print("modelo:", agente.model.model)
print("herramientas:", [t.__name__ for t in agente.tools])
print()
print(agente.instruction)

nombre: asistente_teresa
modelo: ollama_chat/qwen3:4b
herramientas: ['buscar_personaje', 'calcular', 'consultar_sesion']

Eres el asistente del curso de Inteligencia Artificial Aplicada de la Universidad Anáhuac.
Respondes en español, con frases breves y claras y sin emojis, sobre dos temas: la telenovela Teresa (2010) y el calendario del curso.

Reglas:
1. Si te preguntan por un personaje, un actor o una actriz de Teresa, llama a buscar_personaje. No respondas de memoria.
2. Si te preguntan por la fecha, el tema o los días que faltan para una sesión, llama a consultar_sesion con el número de sesión.
3. Para cualquier operación aritmética llama a calcular; nunca hagas cuentas de memoria. Si la operación depende de un dato que da otra herramienta, primero llama a esa herramienta y después a calcular con el dato.
4. Si el usuario solo saluda o pregunta qué sabes hacer, responde sin herramientas.
5. Basa tu respuesta final en los resultados de las herramientas y menciona los datos concret

## 5. `Runner`, sesión y eventos

El `Runner` ejecuta el bucle ReAct. Para cada mensaje del usuario:

1. arma la petición (instrucción, historial de la sesión, esquemas);
2. llama al modelo;
3. si el modelo pide herramientas, las ejecuta y vuelve al paso 1;
4. si el modelo responde en texto, termina.

Cada paso queda como un **evento** (`Event`) con autor, contenido (texto, `function_call` o `function_response`), cambios de estado (`actions.state_delta`) y tokens. El `SessionService` guarda la sesión: historial de eventos y estado. `InMemorySessionService` la guarda en memoria y se pierde al reiniciar.

In [7]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP, USUARIO = "asistente_teresa", "estudiante"
servicio = InMemorySessionService()
runner = Runner(agent=agente, app_name=APP, session_service=servicio)
sesion = await servicio.create_session(app_name=APP, user_id=USUARIO)
print("sesión:", sesion.id, "estado inicial:", sesion.state)


def describir(ev, t0):
    pref = f"[{ev.timestamp - t0:6.1f} s] {ev.author:18s}"
    for p in (ev.content.parts if ev.content else []):
        if p.thought and p.text:
            print(pref, "PENSAMIENTO:", p.text.strip().replace("\n", " ")[:160], "...")
        elif p.function_call:
            print(pref, "LLAMADA     :", p.function_call.name, json.dumps(p.function_call.args, ensure_ascii=False))
        elif p.function_response:
            print(pref, "OBSERVACIÓN :", json.dumps(p.function_response.response, ensure_ascii=False))
        elif p.text:
            print(pref, "TEXTO       :", p.text.strip())
    if ev.actions and ev.actions.state_delta:
        print(pref, "ESTADO Δ    :", dict(ev.actions.state_delta))
    if ev.usage_metadata:
        u = ev.usage_metadata
        print(pref, f"tokens      : entrada {u.prompt_token_count}, salida {u.candidates_token_count}")


async def preguntar(texto, runner=runner, sesion=sesion):
    print("USUARIO:", texto)
    t0 = time.time()
    eventos = []
    msg = types.Content(role="user", parts=[types.Part(text=texto)])
    async for ev in runner.run_async(user_id=USUARIO, session_id=sesion.id, new_message=msg):
        eventos.append(ev)
        describir(ev, t0)
    print(f"--- {len(eventos)} eventos en {time.time() - t0:.1f} s")
    return eventos

sesión: 8af51f4e-aacc-43a4-9bdd-a010f280868a estado inicial: {}


### Turno 1: una pregunta sin herramientas

Un saludo no requiere datos externos: el modelo responde en el primer paso y el bucle termina. `qwen3` es un modelo con razonamiento; sus tokens de pensamiento llegan como partes con `thought=True`.

In [8]:
ev1 = await preguntar("Hola, ¿qué puedes hacer?")

USUARIO: Hola, ¿qué puedes hacer?


[  13.6 s] asistente_teresa   PENSAMIENTO: Okay, the user said "Hola, ¿qué puedes hacer?" which means "Hello, what can you do?" in Spanish. According to rule 4, if the user only saluda or asks what I can ...
[  13.6 s] asistente_teresa   TEXTO       : Hola, puedo ayudarte con información sobre la telenovela *Teresa* (2010) y el calendario del curso de Inteligencia Artificial Aplicada. ¿En qué necesitas ayuda?
[  13.6 s] asistente_teresa   tokens      : entrada 864, salida 178
--- 1 eventos en 13.6 s


### Turno 2: una herramienta

Ahora el modelo debe pedir `buscar_personaje`. Verás tres eventos: la llamada (autor: el agente), la observación (la respuesta de la función) y la respuesta final, que ya incorpora el dato.

In [9]:
ev2 = await preguntar("¿Quién interpreta a Teresa en la telenovela de 2010?")

USUARIO: ¿Quién interpreta a Teresa en la telenovela de 2010?


[  13.4 s] asistente_teresa   PENSAMIENTO: Okay, the user is asking who interprets Teresa in the 2010 telenovela. Let me check the rules.  Rule 1 says if they ask about a character, actor, or actress, I  ...
[  13.4 s] asistente_teresa   LLAMADA     : buscar_personaje {"nombre": "Teresa"}
[  13.4 s] asistente_teresa   tokens      : entrada 939, salida 194
[  13.4 s] asistente_teresa   OBSERVACIÓN : {"status": "ok", "nombre": "Teresa Chávez", "actor": "Angelique Boyer", "rol": "Protagonista", "descripcion": "Joven brillante y ambiciosa de origen humilde que decide usar su belleza e inteligencia para escapar de la pobreza.", "total_personajes_consultados": 1}
[  13.4 s] asistente_teresa   ESTADO Δ    : {'personajes_consultados': ['Teresa Chávez']}


[  43.5 s] asistente_teresa   PENSAMIENTO: Okay, let's see. The user asked who interprets Teresa in the 2010 telenovela. I called the buscar_personaje function with "Teresa" as the name. The response say ...
[  43.5 s] asistente_teresa   TEXTO       : En la telenovela *Teresa* (2010), el personaje Teresa Chávez es interpretado por Angelique Boyer, quien se desempeña como protagonista en la historia.
[  43.5 s] asistente_teresa   tokens      : entrada 1230, salida 405
--- 3 eventos en 43.5 s


### Turno 3: dos herramientas encadenadas

Para convertir a horas los días que faltan, el modelo necesita primero el dato de `consultar_sesion` y luego `calcular`. Es un bucle ReAct de dos vueltas: pensar, actuar, observar, pensar otra vez, actuar, observar, responder.

In [10]:
ev3 = await preguntar("¿Cuántos días faltan para la sesión 8 del curso? Convierte esos días a horas.")

USUARIO: ¿Cuántos días faltan para la sesión 8 del curso? Convierte esos días a horas.


[ 301.1 s] asistente_teresa   PENSAMIENTO: Okay, the user is asking how many days are left until session 8 of the course, and then they want to convert those days into hours. Let me think about how to ap ...
[ 301.1 s] asistente_teresa   LLAMADA     : consultar_sesion {"numero": 8}
[ 301.1 s] asistente_teresa   tokens      : entrada 1136, salida 911
[ 301.1 s] asistente_teresa   OBSERVACIÓN : {"status": "ok", "numero": 8, "fecha": "2026-11-21", "tema": "Creación de un agente simple con Google ADK", "hoy": "2026-11-14", "dias_restantes": 7}
[ 301.1 s] asistente_teresa   ESTADO Δ    : {'ultima_sesion_consultada': 8}


[ 428.6 s] asistente_teresa   PENSAMIENTO: Okay, let me break this down. The user asked how many days are left for session 8 and then convert those days to hours.  First, I called the consultar_sesion fu ...
[ 428.6 s] asistente_teresa   LLAMADA     : calcular {"expresion": "7 * 24"}
[ 428.6 s] asistente_teresa   tokens      : entrada 2131, salida 413
[ 428.6 s] asistente_teresa   OBSERVACIÓN : {"status": "ok", "expresion": "7 * 24", "resultado": 168}


[ 563.4 s] asistente_teresa   PENSAMIENTO: Okay, let me go through the user's question again to make sure I didn't miss anything. The user asked, "¿Cuántos días faltan para la sesión 8 del curso? Convier ...
[ 563.4 s] asistente_teresa   TEXTO       : Para la sesión 8 del curso faltan 7 días, lo que equivale a 168 horas.
[ 563.4 s] asistente_teresa   tokens      : entrada 2582, salida 473
--- 5 eventos en 563.4 s


## 6. Estado de sesión como memoria

El historial de mensajes ya es una memoria, pero crece y se mezcla con todo lo demás. El **estado** es un diccionario explícito asociado a la sesión: las herramientas lo escriben con `tool_context.state[...] = ...` y ADK registra cada cambio en `actions.state_delta`.

La instrucción del agente contiene `{personajes_consultados?}`. Antes de cada llamada al modelo, ADK sustituye esa plantilla por el valor actual del estado; el signo `?` evita un error cuando la clave todavía no existe.

In [11]:
ev4 = await preguntar("¿Y qué actor hace de Arturo de la Barrera?")
actual = await servicio.get_session(app_name=APP, user_id=USUARIO, session_id=sesion.id)
print("\nestado de la sesión:", actual.state)

USUARIO: ¿Y qué actor hace de Arturo de la Barrera?


[  47.0 s] asistente_teresa   PENSAMIENTO: Okay, the user is asking who plays Arturo de la Barrera in the 2010 telenovela Teresa. Let me check the tools I have.  First, I remember that there's a function ...
[  47.0 s] asistente_teresa   LLAMADA     : buscar_personaje {"nombre": "Arturo"}
[  47.0 s] asistente_teresa   tokens      : entrada 1352, salida 447
[  47.1 s] asistente_teresa   OBSERVACIÓN : {"status": "ok", "nombre": "Arturo de la Barrera", "actor": "Sebastián Rulli", "rol": "Protagonista", "descripcion": "Profesor universitario de buena posición que se enamora de Teresa.", "total_personajes_consultados": 2}
[  47.1 s] asistente_teresa   ESTADO Δ    : {'personajes_consultados': ['Teresa Chávez', 'Arturo de la Barrera']}


[  67.3 s] asistente_teresa   PENSAMIENTO: Okay, the user asked who plays Arturo de la Barrera in the 2010 telenovela Teresa. I called the buscar_personaje function with "Arturo" as the name. The respons ...
[  67.3 s] asistente_teresa   TEXTO       : En la telenovela *Teresa* (2010), el personaje Arturo de la Barrera es interpretado por Sebastián Rulli, quien se desempeña como protagonista en la historia.
[  67.3 s] asistente_teresa   tokens      : entrada 1887, salida 298
--- 3 eventos en 67.3 s

estado de la sesión: {'personajes_consultados': ['Teresa Chávez', 'Arturo de la Barrera'], 'ultima_sesion_consultada': 8}


In [12]:
ev5 = await preguntar("¿Qué personajes te he preguntado hasta ahora en esta conversación?")

USUARIO: ¿Qué personajes te he preguntado hasta ahora en esta conversación?


[  71.0 s] asistente_teresa   PENSAMIENTO: Okay, the user is asking which characters I've been asked about in this conversation so far. Let me check the conversation history.  First, the user started by  ...
[  71.0 s] asistente_teresa   TEXTO       : Hasta ahora has preguntado por los personajes: Teresa Chávez y Arturo de la Barrera.
[  71.0 s] asistente_teresa   tokens      : entrada 1536, salida 428
--- 1 eventos en 71.0 s


### La sesión completa

Todos los eventos de los cinco turnos quedan guardados en la sesión. Contamos cuántos son de cada tipo y cuántos tokens consumió el modelo.

In [13]:
import pandas as pd

actual = await servicio.get_session(app_name=APP, user_id=USUARIO, session_id=sesion.id)
filas = []
for ev in actual.events:
    for p in (ev.content.parts if ev.content else []):
        tipo = "function_call" if p.function_call else "function_response" if p.function_response else "pensamiento" if p.thought else "texto"
        filas.append({"autor": ev.author, "tipo": tipo,
                      "tokens_entrada": ev.usage_metadata.prompt_token_count if ev.usage_metadata else None,
                      "tokens_salida": ev.usage_metadata.candidates_token_count if ev.usage_metadata else None})
df = pd.DataFrame(filas)
print(df.groupby(["autor", "tipo"]).size())
consultas = [ev.usage_metadata for ev in actual.events if ev.usage_metadata]
print(f"\n{len(consultas)} consultas al modelo; tokens de entrada:", sum(u.prompt_token_count for u in consultas),
      "; tokens de salida:", sum(u.candidates_token_count for u in consultas))

autor             tipo             
asistente_teresa  function_call        4
                  function_response    4
                  pensamiento          9
                  texto                5
user              texto                5
dtype: int64

9 consultas al modelo; tokens de entrada: 13657 ; tokens de salida: 3747


## 7. Con y sin herramientas

Repetimos la pregunta del turno 3 con un agente idéntico sin herramientas y con la instrucción recortada. El modelo no tiene forma de saber la fecha de hoy ni el calendario: puede negarse o inventar.

In [14]:
sin = modulo.construir_agente(con_herramientas=False)
servicio_sin = InMemorySessionService()
runner_sin = Runner(agent=sin, app_name=APP, session_service=servicio_sin)
sesion_sin = await servicio_sin.create_session(app_name=APP, user_id=USUARIO)
ev_sin = await preguntar("¿Cuántos días faltan para la sesión 8 del curso? Convierte esos días a horas.",
                         runner=runner_sin, sesion=sesion_sin)

USUARIO: ¿Cuántos días faltan para la sesión 8 del curso? Convierte esos días a horas.


[  47.0 s] asistente_sin_herramientas PENSAMIENTO: Okay, the user is asking how many days are left until session 8 of the course, and then wants that converted to hours. Hmm, first I need to check the course cal ...
[  47.0 s] asistente_sin_herramientas TEXTO       : No puedo acceder al calendario actual del curso para calcular los días faltantes. Por favor, consulta el calendario oficial en la plataforma educativa.
[  47.0 s] asistente_sin_herramientas tokens      : entrada 151, salida 809
--- 1 eventos en 47.0 s


## 8. La traza grabada y la interfaz `adk web`

`scripts/grabar_traza_agente.py` repite estas preguntas y guarda todos los eventos, los esquemas y la comparación en `data/processed/traza_agente.json`. Esa traza alimenta la SPA `spa/agente_adk.html` y el apunte de la sesión.

In [15]:
traza = json.loads((ROOT / "data" / "processed" / "traza_agente.json").read_text(encoding="utf-8"))
print("modelo:", traza["modelo"], " fecha de referencia:", traza["hoy"])
for t in traza["turnos"]:
    pasos = [p["nombre"] for e in t["eventos"] for p in e["partes"] if p["tipo"] == "function_call"]
    print(f"{t['id']:17s} {t['duracion_s']:6.1f} s  herramientas: {pasos}")

modelo: ollama_chat/qwen3:4b  fecha de referencia: 2026-11-14
sin_herramientas    22.2 s  herramientas: []
una_herramienta     24.8 s  herramientas: ['buscar_personaje']
encadenada          70.1 s  herramientas: ['consultar_sesion', 'calcular']
estado_escribe      46.5 s  herramientas: ['buscar_personaje']
estado_lee          23.3 s  herramientas: []


ADK trae dos formas de usar el agente sin escribir el `Runner` a mano. Desde la raíz del repositorio y con el túnel activo:

```bash
uv run adk run agentes/asistente_teresa    # conversación en la terminal
uv run adk web agentes                     # interfaz en http://localhost:8000
```

`adk web` busca paquetes dentro de `agentes/` que expongan `root_agent` (por eso `__init__.py` contiene `from . import agent`). En la pestaña **Events** de la interfaz se ve el mismo flujo de eventos de este cuaderno, con cada `function_call` y `function_response`, y en **State** el estado de la sesión.

## 9. Ejercicios

1. Agrega una cuarta herramienta `listar_sesiones()` que devuelva todos los temas del curso. ¿Cambia el número de tokens de entrada de cada petición? Mídelo con `usage_metadata`.
2. Quita la docstring de `calcular` y repite el turno 3. ¿El modelo sigue llamando a la herramienta?
3. Cambia `AGENTE_MODELO` a `ollama_chat/qwen2.5:7b` y repite los cinco turnos. ¿En cuáles se comporta distinto?
4. Haz que `consultar_sesion` acepte también el tema (por ejemplo "difusión") y devuelva el número de sesión. Ajusta los type hints para que el esquema lo refleje.
5. Usa `before_tool_callback` para impedir que `calcular` reciba expresiones de más de 50 caracteres y observa qué hace el modelo con el rechazo.